<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/mimic_iv_rag_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trích xuất thông tin & So sánh hồ sơ tương đòng từ bộ MIMIC-IV dể giải quyết bài toán Dự đoán mức độ sinh tồn**

In [7]:
# ── CELL 1: Cài thư viện + Mount Drive ──────────────────────
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
✓ Libraries loaded


## **Cấu hình đường dẫn**

In [10]:
# ── CELL 2: Đường dẫn dữ liệu ───────────────────────────────
# === CHỈNH Ở ĐÂY ===
# DATA_DIR = r"E:\KLTN\mimiciv\3.1\parquet"  # Windows local
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## **Nạp dữ liệu cơ bản**

In [13]:
# ── CELL 3: Load các bảng chính ───────────────────────────────
print("Loading core tables...")

# hosp
patients   = load("hosp", "patients.csv.gz")
admissions = load("hosp", "admissions.csv.gz")
labevents = load("hosp", "labevents.csv.gz")

# icu
icustays   = load("icu",  "icustays.csv.gz")
chartevents   = load("icu",  "chartevents.csv.gz")
prescriptions   = load("icu",  "prescriptions.csv.gz")
inputevents    = load("icu",  "inputevents.csv.gz")
outputevents   = load("icu",  "outputevents.csv.gz")


# Merge thành cohort
cohort = (
    icustays
    .merge(admissions[["subject_id", "hadm_id", "admittime", "dischtime",
                        "hospital_expire_flag", "insurance", "race"]],
           on=["subject_id", "hadm_id"], how="inner")
    .merge(patients[["subject_id", "gender", "anchor_age", "dod"]],
           on="subject_id", how="inner")
)
# Chỉ người lớn, first ICU stay per admission
cohort = cohort[cohort["anchor_age"] >= 18]
cohort = (cohort.sort_values(["subject_id", "hadm_id", "intime"])
                .groupby(["subject_id", "hadm_id"], as_index=False).first())

# Tính các biến dẫn xuất
cohort["icu_los_hours"] = (
    pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
).dt.total_seconds() / 3600                                                       # Thời gian nằm ICU theo giờ
cohort["hosp_los_days"] = (
    pd.to_datetime(cohort["dischtime"]) - pd.to_datetime(cohort["admittime"])
).dt.total_seconds() / 86400                                                      # Thời gian nằm viện theo ngày
cohort["mortality"] = cohort["hospital_expire_flag"].astype(int)                  # Nhãn mục tiêu
cohort["age_group"] = pd.cut(cohort["anchor_age"],
                              bins=[17, 30, 50, 65, 80, 120],
                              labels=["18-30", "31-50", "51-65", "66-80", "80+"]) # Phân loại độ tuổi thành 5 nhóm
print(f"\nCohort: {len(cohort):,} ICU stays | "
      f"{cohort['subject_id'].nunique():,} unique patients")
print(f"Mortality: {cohort['mortality'].sum():,} ({cohort['mortality'].mean()*100:.1f}%)")

Loading core tables...
  ✓ hosp/patients.csv.gz: 100 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols
  ✓ hosp/labevents.csv.gz: 107,727 rows × 16 cols
  ✓ icu/icustays.csv.gz: 140 rows × 8 cols
  ✓ icu/chartevents.csv.gz: 668,862 rows × 11 cols
  ✗ icu/prescriptions.csv.gz: KHÔNG TÌM THẤY
  ✓ icu/inputevents.csv.gz: 20,404 rows × 26 cols
  ✓ icu/outputevents.csv.gz: 9,362 rows × 9 cols

Cohort: 128 ICU stays | 100 unique patients
Mortality: 15 (11.7%)
